In [1]:
import os 
import glob
from pathlib import Path

current_dir = Path.cwd()
data_dir = current_dir.parent/'data'
viz_dir = current_dir.parent/'viz'
processed_data_dir = current_dir.parent/'processed_data'


os.makedirs(data_dir, exist_ok=True)
os.makedirs(viz_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

In [3]:
from get_opendata import get_data
from pprint import pprint

pprint(
    help(get_data)
)

Help on function get_data in module get_opendata.get_opendata:

get_data(dataset_id, endpoint, datetime_field=None, start_date=None, end_date=None, order=None, first_filter_field=None, first_filter_value=None, second_filter_field=None, second_filter_value=None, third_filter_field=None, third_filter_value=None, app_token=None, base_sleep=5, limit=100000, max_retries=3, backoff_factor=2)

None


In [6]:
dataset_id = 'c7ck-438e'
endpoint = 'data.cityofchicago.org/'

iucr_response = get_data(dataset_id = dataset_id, 
                    endpoint = endpoint)

print('Done')

Getting page 1 with offset 0...
Obtained page 1, records in this page: 434, total records: 434
All data obtained.
Finished fetching all data!
Done


In [9]:
import pandas as pd

iucr_crosswalk = pd.DataFrame(iucr_response)
iucr_crosswalk.columns = iucr_crosswalk.columns.str.lower().str.replace(' ', '_')

iucr_crosswalk.head()

,iucr,primary_description,secondary_description,index_code,active
0,0110,HOMICIDE,FIRST DEGREE MURDER,I,True
1,0130,HOMICIDE,SECOND DEGREE MURDER,I,True
2,0141,HOMICIDE,INVOLUNTARY MANSLAUGHTER,N,True
3,0142,HOMICIDE,RECKLESS HOMICIDE,N,True
4,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,I,True


In [16]:
iucr_crosswalk[
   (iucr_crosswalk.primary_description.str.contains('sex', case=False, na=False)) |
   ( iucr_crosswalk.primary_description.str.contains('rape', case=False, na=False)) 
    ].shape

(34, 5)

In [26]:
sexCrime_wordList = ['sex', 'rape']

iucr_sexCrimes_crosswalk = iucr_crosswalk[
   (iucr_crosswalk.primary_description.str.contains('|'.join(sexCrime_wordList), case=False, na=False)) |
   ( iucr_crosswalk.secondary_description.str.contains('|'.join(sexCrime_wordList), case=False, na=False)) 
]

iucr_sexCrimes_crosswalk.head()

,iucr,primary_description,secondary_description,index_code,active
4,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,I,True
5,0262,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - OTHER FIREARM,I,True
6,0263,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - KNIFE / CUTTING INSTRUMENT,I,True
7,0264,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - OTHER DANGEROUS WEAPON,I,True
8,0265,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - OTHER,I,True


In [25]:
iucr_sexCrimes_codeList= iucr_sexCrimes_crosswalk.iucr.to_list()
len(iucr_sexCrimes_codeList)

iucr_sexCrimes_codeList[0]

'0261'

In [36]:
dataset_id = 'ijzp-q8t2'
endpoint = 'data.cityofchicago.org/'
datetime_field= 'date'
start_date = '2026-01-01'
end_date= '2026-10-29'
first_filter_field= 'IUCR'
first_filter_value= '0261'
app_token='4kTL9oUmhmB0off4B3U9Z5BqZ',


response = get_data(dataset_id = dataset_id, 
                    endpoint = endpoint,
                    datetime_field=datetime_field, 
                    start_date=start_date, 
                    end_date=end_date,
                    first_filter_field =first_filter_field, 
                    first_filter_value = first_filter_value,
                    app_token='4kTL9oUmhmB0off4B3U9Z5BqZ'
                    )

Getting page 1 with offset 0...
Obtained page 1, records in this page: 32, total records: 32
All data obtained.
Finished fetching all data!


In [27]:
df = pd.DataFrame(response)
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14340014,JK428274,2026-09-24T10:00:00.000,046XX W HARRISON ST,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,ALLEY,False,False,...,24,25,02,1145614,1896954,2026,2026-10-02T15:42:49.000,41.87323571,-87.740850262,"{'latitude': '41.87323571', 'longitude': '-87...."
1,14340745,JK429026,2026-09-27T01:30:00.000,009XX N PULASKI RD,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,ALLEY,False,False,...,37,23,02,1149548,1906070,2026,2026-10-05T15:42:09.000,41.898175582,-87.726169689,"{'latitude': '41.898175582', 'longitude': '-87..."
2,14272148,JK345551,2026-07-24T00:00:00.000,015XX W 78TH ST,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,APARTMENT,True,False,...,17,71,02,1167323,1853003,2026,2026-10-07T15:42:47.000,41.752190503,-87.662405734,"{'latitude': '41.752190503', 'longitude': '-87..."
3,14359159,JK451638,2026-10-15T03:45:00.000,051XX W MADISON ST,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,ALLEY,False,True,...,28,25,02,1142285,1899542,2026,2026-10-23T15:42:49.000,41.880399914,-87.753008553,"{'latitude': '41.880399914', 'longitude': '-87..."
4,14367841,JK462156,2026-10-19T21:00:00.000,018XX W 79TH ST,0261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,APARTMENT,False,False,...,17,71,02,1165558,1852295,2026,2026-10-27T15:52:53.000,41.750285238,-87.668893737,"{'latitude': '41.750285238', 'longitude': '-87..."


In [43]:
help(get_data)

Help on function get_data in module get_opendata.get_opendata:

get_data(dataset_id, endpoint, datetime_field=None, start_date=None, end_date=None, order=None, first_filter_field=None, first_filter_value=None, second_filter_field=None, second_filter_value=None, third_filter_field=None, third_filter_value=None, app_token=None, base_sleep=5, limit=100000, max_retries=3, backoff_factor=2)



In [44]:
dataset_id = 'ijzp-q8t2'
endpoint = 'data.cityofchicago.org/'
datetime_field= 'date'
start_date = '2016-01-01'
end_date= '2026-10-29'
first_filter_field= 'IUCR'
max_retries=10
backoff_factor=5

resopnse_list = []
for iucr_sexCrimes_code in iucr_sexCrimes_codeList:
    first_filter_value= iucr_sexCrimes_code

   
    response = get_data(dataset_id = dataset_id, 
                    endpoint = endpoint,
                    datetime_field=datetime_field, 
                    start_date=start_date, 
                    end_date=end_date,
                    first_filter_field =first_filter_field, 
                    first_filter_value = first_filter_value,
                    app_token='4kTL9oUmhmB0off4B3U9Z5BqZ',
                    max_retries=max_retries,
                    backoff_factor=backoff_factor
    )
    
    print(f'Finished fetching {iucr_sexCrimes_code}')
    print(f'--------')

    resopnse_list.extend(response)

print('Done!')

Getting page 1 with offset 0...
[Retry 1/10] HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=10)
Retrying in 5 seconds...
Getting page 1 with offset 0...
Obtained page 1, records in this page: 618, total records: 618
All data obtained.
Finished fetching all data!
Finished fetching 0261
--------
Getting page 1 with offset 0...
Obtained page 1, records in this page: 20, total records: 20
All data obtained.
Finished fetching all data!
Finished fetching 0262
--------
Getting page 1 with offset 0...
Obtained page 1, records in this page: 301, total records: 301
All data obtained.
Finished fetching all data!
Finished fetching 0263
--------
Getting page 1 with offset 0...
Obtained page 1, records in this page: 121, total records: 121
All data obtained.
Finished fetching all data!
Finished fetching 0264
--------
Getting page 1 with offset 0...
Obtained page 1, records in this page: 2431, total records: 2431
All data obtained.
Finished fetching all da

In [45]:
len(resopnse_list)

43209

In [46]:
df = pd.DataFrame(resopnse_list)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43209 entries, 0 to 43208
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    43209 non-null  object
 1   case_number           43209 non-null  object
 2   date                  43209 non-null  object
 3   block                 43209 non-null  object
 4   iucr                  43209 non-null  object
 5   primary_type          43209 non-null  object
 6   description           43209 non-null  object
 7   location_description  43184 non-null  object
 8   arrest                43209 non-null  bool  
 9   domestic              43209 non-null  bool  
 10  beat                  43209 non-null  object
 11  district              43209 non-null  object
 12  ward                  43207 non-null  object
 13  community_area        43207 non-null  object
 14  fbi_code              43209 non-null  object
 15  x_coordinate          38170 non-null

In [51]:
df.groupby('iucr')\
    .primary_type.value_counts(dropna=False)

iucr  primary_type              
0261  CRIMINAL SEXUAL ASSAULT        444
      CRIM SEXUAL ASSAULT            174
0262  CRIMINAL SEXUAL ASSAULT         19
      CRIM SEXUAL ASSAULT              1
0263  CRIMINAL SEXUAL ASSAULT        209
      CRIM SEXUAL ASSAULT             92
0264  CRIMINAL SEXUAL ASSAULT         74
      CRIM SEXUAL ASSAULT             47
0265  CRIMINAL SEXUAL ASSAULT       1777
      CRIM SEXUAL ASSAULT            654
0266  CRIMINAL SEXUAL ASSAULT        807
      CRIM SEXUAL ASSAULT            386
0271  CRIMINAL SEXUAL ASSAULT         42
      CRIM SEXUAL ASSAULT             16
0272  CRIMINAL SEXUAL ASSAULT          2
      CRIM SEXUAL ASSAULT              1
0273  CRIMINAL SEXUAL ASSAULT         20
      CRIM SEXUAL ASSAULT             14
0274  CRIMINAL SEXUAL ASSAULT          6
      CRIM SEXUAL ASSAULT              2
0275  CRIMINAL SEXUAL ASSAULT         82
      CRIM SEXUAL ASSAULT             41
0281  CRIMINAL SEXUAL ASSAULT       9242
      CRIM SEXUAL ASSAUL

In [48]:
df.iucr.nunique()

41

In [49]:
saved_filename = 'chicago_sexCrimes'

df.to_parquet(
    f'{data_dir}/{saved_filename}.parquet'
)